In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler,PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import SGDRegressor

In [2]:
df = pd.read_csv(r"C:\Users\udays\Documents\Mercedes\Model\mercedes_benz_sales.csv")

In [3]:
df.head()

,Model,Year,Region,Color,Fuel Type,Base Price (USD),Horsepower,Sales Volume,Turbo
0,A-Class,2020,Global,Yellow,Diesel,41265,252,1,Yes
1,A-Class,2020,Global,Black,Petrol,51023,249,1,No
2,A-Class,2020,Global,Grey,Petrol,72819,341,1,Yes
3,A-Class,2020,Global,Black,Petrol,62480,385,1,Yes
4,A-Class,2020,Global,White,Petrol,35189,337,1,Yes


In [4]:
df.shape

(12132666, 9)

In [5]:
df.describe()

,Year,Base Price (USD),Horsepower,Sales Volume
count,1.213267e+07,1.213267e+07,1.213267e+07,12132666.0
mean,2.022548e+03,1.042169e+05,3.674594e+02,1.0
std,1.658896e+00,6.783388e+04,1.195446e+02,0.0
min,2.020000e+03,3.500000e+04,1.500000e+02,1.0
25%,2.021000e+03,6.346200e+04,2.750000e+02,1.0
50%,2.023000e+03,8.357200e+04,3.580000e+02,1.0
75%,2.024000e+03,1.199200e+05,4.440000e+02,1.0
max,2.025000e+03,4.112460e+05,8.310000e+02,1.0


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12132666 entries, 0 to 12132665
Data columns (total 9 columns):
 #   Column            Dtype 
---  ------            ----- 
 0   Model             object
 1   Year              int64 
 2   Region            object
 3   Color             object
 4   Fuel Type         object
 5   Base Price (USD)  int64 
 6   Horsepower        int64 
 7   Sales Volume      int64 
 8   Turbo             object
dtypes: int64(4), object(5)
memory usage: 833.1+ MB


In [7]:
cat_col = df.select_dtypes(include=["object"]).columns

In [8]:
for col in cat_col:
    print(f"{col} : {df[col].unique()}")

Model : ['A-Class' 'C-Class' 'E-Class' 'S-Class' 'CLA' 'CLS' 'GLA' 'GLB' 'GLC'
 'GLE' 'GLS' 'G-Class' 'AMG GT' 'AMG A 45' 'AMG C 63' 'AMG E 63'
 'AMG S 63']
Region : ['Global']
Color : ['Yellow' 'Black' 'Grey' 'White' 'Silver' 'Brown' 'Blue' 'Red' 'Green'
 'Orange']
Fuel Type : ['Diesel' 'Petrol' 'Hybrid' 'Electric']
Turbo : ['Yes' 'No']


In [9]:
df = df.loc[:,df.nunique()>1] 

In [10]:
df.head()

,Model,Year,Color,Fuel Type,Base Price (USD),Horsepower,Turbo
0,A-Class,2020,Yellow,Diesel,41265,252,Yes
1,A-Class,2020,Black,Petrol,51023,249,No
2,A-Class,2020,Grey,Petrol,72819,341,Yes
3,A-Class,2020,Black,Petrol,62480,385,Yes
4,A-Class,2020,White,Petrol,35189,337,Yes


In [11]:
df.nunique()

Model                   17
Year                     6
Color                   10
Fuel Type                4
Base Price (USD)    366227
Horsepower             682
Turbo                    2
dtype: int64

In [12]:
X = df.drop(columns=["Base Price (USD)"])
y = df["Base Price (USD)"]

In [ ]:
X_trainval,X_test,y_trainval,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [14]:
X_train,X_val,y_train,y_val = train_test_split(X_trainval,y_trainval,test_size=0.25,random_state=42)

In [15]:
X_train["Model"].unique()

array(['E-Class', 'GLA', 'C-Class', 'GLC', 'GLS', 'A-Class', 'GLB', 'CLA',
       'G-Class', 'S-Class', 'GLE', 'AMG E 63', 'AMG C 63', 'AMG A 45',
       'CLS', 'AMG GT', 'AMG S 63'], dtype=object)

In [15]:
X_train.head()

,Model,Year,Color,Fuel Type,Horsepower,Turbo
6528692,E-Class,2023,Silver,Diesel,485,Yes
11172239,GLA,2025,Black,Hybrid,220,Yes
4217031,C-Class,2022,Yellow,Petrol,327,Yes
4136543,C-Class,2022,Orange,Diesel,319,Yes
9643857,GLC,2024,Grey,Hybrid,208,Yes


In [16]:
ohe_encoder = Pipeline([ ("ohe_encoding",OneHotEncoder(drop="first",handle_unknown = "ignore")) ])
numeric_pipe = Pipeline([
    ("polynomial",PolynomialFeatures(degree=2,include_bias=False)),
    ("Scale",StandardScaler())
])

In [18]:
col = ColumnTransformer(transformers=[
    ("OHE",ohe_encoder,["Model","Color","Fuel Type","Turbo"]),
    ("Numeric",numeric_pipe,["Horsepower","Year"])
],remainder="passthrough")

In [19]:
model = Pipeline(steps=[
    ("preprocessing",col),
    ("model",SGDRegressor(
        loss="squared_error",
        penalty="l2",
        alpha = 0.0001,
        eta0=0.1,
        max_iter = 1000,
        early_stopping = True,
        learning_rate = "adaptive",
        random_state=42,
        verbose = 1
    ))
])

In [20]:
model.fit(X_train,y_train)

-- Epoch 1
Norm: 557879.96, NNZs: 34, Bias: 70870.393964, T: 6551639, Avg. loss: 1998750939.198111
Total training time: 6.28 seconds.
-- Epoch 2
Norm: 519709.90, NNZs: 34, Bias: 71338.814824, T: 13103278, Avg. loss: 1874128219.962757
Total training time: 13.14 seconds.
-- Epoch 3
Norm: 562921.81, NNZs: 34, Bias: 68907.015494, T: 19654917, Avg. loss: 1997757150.784433
Total training time: 19.59 seconds.
-- Epoch 4
Norm: 529796.55, NNZs: 34, Bias: 70328.019028, T: 26206556, Avg. loss: 1855193658.187289
Total training time: 26.48 seconds.
-- Epoch 5
Norm: 518878.44, NNZs: 34, Bias: 71155.132912, T: 32758195, Avg. loss: 1949465220.259533
Total training time: 33.27 seconds.
-- Epoch 6
Norm: 495031.90, NNZs: 34, Bias: 68793.716803, T: 39309834, Avg. loss: 2022457044.216025
Total training time: 40.25 seconds.
-- Epoch 7
Norm: 527888.97, NNZs: 34, Bias: 68998.741274, T: 45861473, Avg. loss: 595479030.810331
Total training time: 47.30 seconds.
-- Epoch 8
Norm: 518100.40, NNZs: 34, Bias: 68892.6

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('OHE',
                                                  Pipeline(steps=[('ohe_encoding',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore'))]),
                                                  ['Model', 'Color',
                                                   'Fuel Type', 'Turbo']),
                                                 ('Numeric',
                                                  Pipeline(steps=[('polynomial',
                                                                   PolynomialFeatures(include_bias=False)),
                                                                  ('Scale',
                                                                   StandardScaler())]),
                                                  ['Horsepower', 'Year'])])),
                ('model',
                 SGDRegressor(early_stopping=True, eta0=0.1,
                              learning_rate='adaptive', random_state=42,
                              verbose=1))])

In [21]:
model.score(X_test,y_test)

0.7724026019867128

In [22]:
joblib.dump(model,"model")

['model']